In [1]:
import warnings
warnings.filterwarnings("ignore")
import tensorflow as tf
import matplotlib.pyplot as plt
tf.compat.v1.set_random_seed(0)
from tensorflow import keras
import numpy as np
np.random.seed(0)
import itertools
from keras.preprocessing.image import image_dataset_from_directory
from tensorflow.keras.layers.experimental.preprocessing import Rescaling
from sklearn.metrics import precision_score, accuracy_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

In [2]:

train_gen = image_dataset_from_directory(directory="../input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train",
                                         image_size=(256, 256))
test_gen = image_dataset_from_directory(directory="../input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid",
                                        image_size=(256, 256))

rescale = Rescaling(scale=1.0/255)
train_gen = train_gen.map(lambda image,label:(rescale(image),label))
test_gen  = test_gen.map(lambda image,label:(rescale(image),label))

Found 70295 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.


## Simple CNN

In [4]:
model = keras.Sequential()

model.add(keras.layers.Conv2D(32,(3,3),activation="relu",padding="same",input_shape=(256,256,3)))
model.add(keras.layers.Conv2D(32,(3,3),activation="relu",padding="same"))
model.add(keras.layers.MaxPooling2D(3,3))

model.add(keras.layers.Conv2D(64,(3,3),activation="relu",padding="same"))
model.add(keras.layers.Conv2D(64,(3,3),activation="relu",padding="same"))
model.add(keras.layers.MaxPooling2D(3,3))

model.add(keras.layers.Conv2D(128,(3,3),activation="relu",padding="same"))
model.add(keras.layers.Conv2D(128,(3,3),activation="relu",padding="same"))
model.add(keras.layers.MaxPooling2D(3,3))

model.add(keras.layers.Conv2D(256,(3,3),activation="relu",padding="same"))
model.add(keras.layers.Conv2D(256,(3,3),activation="relu",padding="same"))

model.add(keras.layers.Conv2D(512,(5,5),activation="relu",padding="same"))
model.add(keras.layers.Conv2D(512,(5,5),activation="relu",padding="same"))

model.add(keras.layers.Flatten())

model.add(keras.layers.Dense(1568,activation="relu"))
model.add(keras.layers.Dropout(0.5))

model.add(keras.layers.Dense(38,activation="softmax"))

opt = keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=opt,loss="sparse_categorical_crossentropy",metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 256, 256, 32)      896       
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 256, 256, 32)      9248      
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 85, 85, 32)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 85, 85, 64)        18496     
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 85, 85, 64)        36928     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 28, 28, 64)        0         
_________________________________________________________________
conv2d_4 (Conv2D)            (None, 28, 28, 128)       7

In [ ]:
ep = 10
history = model.fit_generator(train_gen,
          validation_data=test_gen,
          epochs = ep)

In [ ]:
plt.figure(figsize = (20,5))
plt.subplot(1,2,1)
plt.title("Train and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.plot(history.history['loss'],label="Train Loss")
plt.plot(history.history['val_loss'], label="Validation Loss")
plt.xlim(0, 10)
plt.ylim(0.0,1.0)
plt.legend()

plt.subplot(1,2,2)
plt.title("Train and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.plot(history.history['accuracy'], label="Train Accuracy")
plt.plot(history.history['val_accuracy'], label="Validation Accuracy")
plt.xlim(0, 9.25)
plt.ylim(0.75,1.0)
plt.legend()
plt.tight_layout()

In [ ]:
labels = []
predictions = []
for x,y in test_gen:
    labels.append(list(y.numpy()))
    predictions.append(tf.argmax(model.predict(x),1).numpy())

In [ ]:
predictions = list(itertools.chain.from_iterable(predictions))
labels = list(itertools.chain.from_iterable(labels))

In [ ]:
print("Train Accuracy      : {:.2f} %".format(history.history['accuracy'][-1]*100))
print("Validation Accuracy : {:.2f} %".format(history.history['val_accuracy'][-1]*100))

print("Test Accuracy       : {:.2f} %".format(accuracy_score(labels, predictions) * 100))
print("Precision Score     : {:.2f} %".format(precision_score(labels, predictions, average='micro') * 100))
print("Recall Score        : {:.2f} %".format(recall_score(labels, predictions, average='micro') * 100))

In [ ]:
plt.figure(figsize= (20,5))
cm = confusion_matrix(labels, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=list(range(1,39)))
fig, ax = plt.subplots(figsize=(15,15))
disp.plot(ax=ax,colorbar= False,cmap = 'YlGnBu')
plt.title("Confusion Matrix")
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

## HYBRID CNN

In [ ]:

from tensorflow.keras import layers, models

def build_hybrid_cnn():
    inputs = keras.Input(shape=(256,256,3))

   
    x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2,2)(x)

    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2,2)(x)

    x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2,2)(x)

    

    x = layers.GlobalAveragePooling2D()(x)

    
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)

    
    x = layers.Dense(128, activation='tanh')(x)

    # ===== OUTPUT =====
    outputs = layers.Dense(38, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model


# Build model
hybrid_model = build_hybrid_cnn()

# Compile
hybrid_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

hybrid_model.summary()

In [ ]:
ep = 10

history_hybrid = hybrid_model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=ep
)

In [ ]:
plt.figure(figsize=(20,5))

plt.subplot(1,2,1)
plt.title("Hybrid CNN Loss")
plt.plot(history_hybrid.history['loss'], label="Train Loss")
plt.plot(history_hybrid.history['val_loss'], label="Val Loss")
plt.legend()

plt.subplot(1,2,2)
plt.title("Hybrid CNN Accuracy")
plt.plot(history_hybrid.history['accuracy'], label="Train Acc")
plt.plot(history_hybrid.history['val_accuracy'], label="Val Acc")
plt.legend()

plt.show()

In [ ]:
# ===== GET TRUE LABELS AND PREDICTIONS =====

predictions = []
labels = []

for x, y in test_gen:
    pred = hybrid_model.predict(x)
    pred = np.argmax(pred, axis=1)

    predictions.extend(pred)
    labels.extend(y.numpy())

predictions = np.array(predictions)
labels = np.array(labels)

In [ ]:
print("Train Accuracy  : {:.2f} %".format(history_hybrid.history['accuracy'][-1]*100))
print("Validation Accuracy : {:.2f} %".format(history_hybrid.history['val_accuracy'][-1]*100))

print("Test Accuracy   : {:.2f} %".format(accuracy_score(labels, predictions) * 100))
print("Precision Score : {:.2f} %".format(precision_score(labels, predictions, average='micro') * 100))
print("Recall Score    : {:.2f} %".format(recall_score(labels, predictions, average='micro') * 100))

In [ ]:
plt.figure(figsize=(20,5))

cm = confusion_matrix(labels, predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(range(1,39))
)

fig, ax = plt.subplots(figsize=(15,15))
disp.plot(ax=ax, colorbar=False, cmap='YlGnBu')

plt.title("Hybrid CNN Confusion Matrix")
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

## QUANTUM CNN


In [ ]:


from tensorflow.keras import layers, models

def build_qcnn():

    inputs = keras.Input(shape=(256,256,3))

    # ===== CNN FEATURE EXTRACTION =====
    x = layers.Conv2D(32,(3,3),activation='relu',padding='same')(inputs)
    x = layers.MaxPooling2D(2,2)(x)

    x = layers.Conv2D(64,(3,3),activation='relu',padding='same')(x)
    x = layers.MaxPooling2D(2,2)(x)

    x = layers.Conv2D(128,(3,3),activation='relu',padding='same')(x)
    x = layers.MaxPooling2D(2,2)(x)

    x = layers.GlobalAveragePooling2D()(x)

    

    x = layers.Dense(64)(x)
    x = tf.math.sin(x)   
    x = layers.Dense(64)(x)
    x = tf.math.cos(x)

    # ===== OUTPUT =====
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(38, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model

In [ ]:
qcnn_model = build_qcnn()

qcnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ep = 10

history_qcnn = qcnn_model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=ep
)

In [ ]:
predictions = []
labels = []

for x, y in test_gen:
    pred = qcnn_model.predict(x)
    pred = np.argmax(pred, axis=1)

    predictions.extend(pred)
    labels.extend(y.numpy())

predictions = np.array(predictions)
labels = np.array(labels)

print("Train Accuracy  : {:.2f} %".format(history_qcnn.history['accuracy'][-1]*100))
print("Validation Accuracy : {:.2f} %".format(history_qcnn.history['val_accuracy'][-1]*100))

print("Test Accuracy   : {:.2f} %".format(accuracy_score(labels, predictions) * 100))
print("Precision Score : {:.2f} %".format(precision_score(labels, predictions, average='micro') * 100))
print("Recall Score    : {:.2f} %".format(recall_score(labels, predictions, average='micro') * 100))

In [ ]:
cm = confusion_matrix(labels, predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(range(1,39))
)

fig, ax = plt.subplots(figsize=(15,15))
disp.plot(ax=ax, colorbar=False, cmap='YlGnBu')

plt.title("Quantum-Inspired CNN Confusion Matrix")
plt.show()

In [ ]:
#Model Comparision

import matplotlib.pyplot as plt
import numpy as np


models = ['Simple CNN', 'Hybrid CNN']

train_acc = [98.29, 95.00]
val_acc   = [95.19, 96.55]   
test_acc  = [95.19, 96.55]
precision = [95.19, 96.55]
recall    = [95.19, 96.55]

x = np.arange(len(models))
width = 0.15   

# ===== BAR GRAPH =====
plt.figure(figsize=(14,6))

plt.bar(x - 2*width, train_acc, width, label='Train Accuracy')
plt.bar(x - width, val_acc, width, label='Validation Accuracy')
plt.bar(x, test_acc, width, label='Test Accuracy')
plt.bar(x + width, precision, width, label='Precision')
plt.bar(x + 2*width, recall, width, label='Recall')

plt.xticks(x, models)
plt.ylabel('Percentage (%)')
plt.title('Model Comparison: CNN vs Hybrid CNN vs Quantum CNN')
plt.legend()

plt.show()

In [ ]:
import pandas as pd

data = {
    "Model": models,
    "Train Accuracy": train_acc,
    "Validation Accuracy": val_acc,   
    "Test Accuracy": test_acc,
    "Precision": precision,
    "Recall": recall
}

df = pd.DataFrame(data)
print(df)

In [ ]:
import pandas as pd

# ===== CREATE DATA =====
data = {
    "Model": ["Simple CNN", "Hybrid CNN"],
    "Train Accuracy (%)": [98.29, 95.00],
    "Validation Accuracy (%)": [95.19, 96.55],
    "Test Accuracy (%)": [95.19, 96.55],
    "Precision (%)": [95.19, 96.55],
    "Recall (%)": [95.19, 96.55]
}

df = pd.DataFrame(data)

# ===== STYLE FUNCTION =====
def highlight_rows(row):
    if row.name % 2 == 0:
        return ['background-color: #f2f2f2; color: black'] * len(row)
    else:
        return ['background-color: #d6eaf8; color: black'] * len(row)

# ===== DISPLAY TABLE =====
styled_df = df.style.apply(highlight_rows, axis=1)\
    .set_properties(**{'text-align': 'center'})\
    .set_table_styles([
        {
            'selector': 'th',
            'props': [
                ('background-color', '#1f4e79'),
                ('color', 'white'),
                ('text-align', 'center'),
                ('font-weight', 'bold')
            ]
        }
    ])

styled_df